# 1 reading the data

In [4]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

In [7]:
conn=sqlite3.connect('customer_churn.db')
q="""select name from sqlite_master where type='table'"""
tables = pd.read_sql(q, conn)

In [8]:
print(tables)

              name
0      db_customer
1  db_subscription
2       db_support


In [9]:
for table_name in tables['name']:
    df=pd.read_sql(f"select * from {table_name}",conn)
    globals()[f'df_{table_name}']=df

In [14]:
display(df_db_customer.head(),df_db_subscription.head(),df_db_support.head())

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaN,NaN,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,NaN
3,0013-MHZWF,2025-03-18 00:00:00,N,90,None,guidance to renew
4,0013-SMEOE,2024-11-01 00:00:00,N,30,None,NaN


# 2 data cleaning


In [16]:
!pip install skimpy


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ------------------------ --------------- 8.4/13.7 MB 41.2 MB/s eta 0:00:01
   ---------------------------------------- 13.7/13.7 MB 34.8 MB/s  0:00:00
   ---------------------------------------- 0.0/837.6 kB ? eta -:--:--
   ---------------------------------------- 837.6/837.6 kB 30.4 MB/s  0:00:00
   ---------------------------------------- 0.0/52.7 MB ? eta -:--:--
   ------- -------------------------------- 10.2/52.7 MB 51.6 MB/s eta 0:00:01
   -------------- ------------------------- 18.9/52.7 MB 46.5 MB/s eta 0:00:01
   -------------------- ------------------- 27.3/52.7 MB 45.0 MB/s eta 0:00:01
   ---------------------------- ----------- 37.0/52.7 MB 45.7 MB/s eta 0:00:01
   -------------------------------- ------- 42.2/52.7 MB 41.5 MB/s eta 0:00:01
   ----------------------------------- ---- 46.4/52.7 MB 37.5 MB/s eta 0:00:01
   --


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from skimpy import skim

skim(df_db_customer)
skim(df_db_subscription)
skim(df_db_support)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 21     │ │ string      │ 7     │                                                          │
│ │ Number of columns │ 8      │ │ object      │ 1     │                                                          │
│ └───────────────────┴────────┘ └─────────────┴───────┘                                                          │
│                                                    All null                                                     │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column                                            ┃ NA                   ┃ NA %                            ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩  │
│ │                      pincode                      │                   21 │                             100 │  │
│ └───────────────────────────────────────────────────┴──────────────────────┴─────────────────────────────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━┳━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓  │
│ ┃           ┃    ┃          ┃           ┃          ┃           ┃          ┃ chars per ┃ words    ┃ total     ┃  │
│ ┃ column    ┃ NA ┃ NA %     ┃ shortest  ┃ longest  ┃ min       ┃ max      ┃ row       ┃ per row  ┃ words     ┃  │
│ ┡━━━━━━━━━━━╇━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩  │
│ │ customeri │  0 │        0 │ 0002-ORFB │ 0002-ORF │ 0002-ORFB │ 0023-UYU │        10 │        1 │        21 │  │
│ │     d     │    │          │     O     │    BO    │     O     │    PN    │           │          │           │  │
│ │   name    │  0 │        0 │   mira    │ raghvend │  Madhav   │ vishakha │      5.86 │        1 │        21 │  │
│ │           │    │          │           │    ra    │           │          │           │          │           │  │
│ │  country  │  3 │ 14.28571 │   India   │  India   │   India   │  Nepal   │         5 │     0.86 │        18 │  │
│ │           │    │ 42857142 │           │          │           │          │           │          │           │  │
│ │           │    │       86 │           │          │           │          │           │          │           │  │
│ │   state   │  0 │        0 │   Delhi   │  Uttar   │   Delhi   │  Uttar   │      8.86 │      1.1 │        23 │  │
│ │           │    │          │           │ Pradesh  │           │ Pradesh  │           │          │           │  │
│ │  gender   │  0 │        0 │    Men    │  Female  │  Female   │  Women   │      4.86 │        1 │        21 │  │
│ │    dob    │  0 │        0 │ 1982-04-1 │ 1982-04- │ 1976-09-2 │ 2004-12- │        19 │        2 │        42 │  │
│ │           │    │          │     2     │    12    │     1     │    01    │           │          │           │  │
│ │           │    │          │ 00:00:00  │ 00:00:00 │ 00:00:00  │ 00:00:00 │           │          │           │  │
│ │ interests │ 17 │ 80.95238 │    job    │  travel  │   drama   │  travel  │      4.75 │     0.19 │         4 │  │
│ │           │    │ 09523809 │           │          │           │          │           │          │           │  │
│ │           │    │        5 │           │          │  

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 21     │ │ string      │ 8     │                                                          │
│ │ Number of columns │ 11     │ │ int64       │ 2     │                                                          │
│ └───────────────────┴────────┘ │ float64     │ 1     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓  │
│ ┃ column              ┃ NA  ┃ NA %  ┃ mean    ┃ sd     ┃ p0    ┃ p25    ┃ p50    ┃ p75    ┃ p100   ┃ hist    ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩  │
│ │   monthly_charges   │   0 │     0 │   18.85 │  17.75 │  6.99 │  12.99 │  13.99 │  20.99 │  92.99 │ ▇▂   ▁  │  │
│ │        cltv         │   0 │     0 │   823.5 │  668.9 │    42 │    240 │    640 │   1150 │   2185 │ ▇▃▃▁▂▃  │  │
│ │     churn_score     │   0 │     0 │   43.29 │  32.29 │     3 │     14 │     34 │     76 │     99 │ ▇▇▃▃▃▅  │  │
│ └─────────────────────┴─────┴───────┴─────────┴────────┴───────┴────────┴────────┴────────┴────────┴─────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━┳━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┓  │
│ ┃           ┃    ┃           ┃          ┃           ┃          ┃           ┃ chars    ┃ words per ┃ total    ┃  │
│ ┃ column    ┃ NA ┃ NA %      ┃ shortest ┃ longest   ┃ min      ┃ max       ┃ per row  ┃ row       ┃ words    ┃  │
│ ┡━━━━━━━━━━━╇━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━┩  │
│ │ customeri │  0 │         0 │ 0002-ORF │ 0002-ORFB │ 0002-ORF │ 0023-UYUP │       10 │         1 │       21 │  │
│ │     d     │    │           │    BO    │     O     │    BO    │     N     │          │           │          │  │
│ │ subscript │  0 │         0 │ 2021-03- │ 2021-03-1 │ 2019-05- │ 2023-09-1 │       10 │         1 │       21 │  │
│ │ ion_start │    │           │    15    │     5     │    10    │     4     │          │           │          │  │
│ │   _date   │    │           │          │           │          │           │          │           │          │  │
│ │ subscript │  0 │         0 │   Paid   │ Refferal  │ Organic  │ Refferal  │     6.43 │         1 │       21 │  │
│ │ ion_type  │    │           │          │           │          │           │          │           │          │  │
│ │ renewal_d │  0 │         0 │ 2025-03- │ 2025-03-1 │ 2024-01- │ 2025-12-3 │       10 │         1 │       21 │  │
│ │    ate    │    │           │    15    │     5     │    05    │     1     │          │           │          │  │
│ │ plan_type │  0 │         0 │  Basic   │ Standard  │  Basic   │ Standard  │     6.95 │         1 │       21 │  │
│ │ contract_ │  0 │         0 │  Annual  │  Monthly  │  Annual  │  Monthly  │     6.43 │         1 │       21 │  │
│ │   type    │    │           │          │           │          │           │          │           │          │  │
│ │ cancellat │ 15 │ 71.428571 │ 2024-09- │ 2024-09-1 │ 

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 9      │ │ string      │ 4     │                                                          │
│ │ Number of columns │ 6      │ │ int64       │ 1     │                                                          │
│ └───────────────────┴────────┘ │ object      │ 1     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                    All null                                                     │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column                                         ┃ NA                     ┃ NA %                             ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩  │
│ │                     col_1                      │                      9 │                              100 │  │
│ └────────────────────────────────────────────────┴────────────────────────┴──────────────────────────────────┘  │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓  │
│ ┃ column           ┃ NA   ┃ NA %    ┃ mean     ┃ sd       ┃ p0   ┃ p25   ┃ p50   ┃ p75   ┃ p100   ┃ hist     ┃  │
│ ┡━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩  │
│ │    csat_score    │    0 │       0 │    40.56 │    31.67 │   10 │    20 │    30 │    60 │     90 │  ▇▇ ▃ ▅  │  │
│ └──────────────────┴──────┴─────────┴──────────┴──────────┴──────┴───────┴───────┴───────┴────────┴──────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━┳━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┓  │
│ ┃           ┃    ┃           ┃          ┃           ┃          ┃           ┃ chars    ┃ words per ┃ total    ┃  │
│ ┃ column    ┃ NA ┃ NA %      ┃ shortest ┃ longest   ┃ min      ┃ max       ┃ per row  ┃ row       ┃ words    ┃  │
│ ┡━━━━━━━━━━━╇━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━┩  │
│ │ customeri │  0 │         0 │ 0003-MKN │ 0003-MKNF │ 0003-MKN │ 0022-TCJC │       10 │         1 │        9 │  │
│ │     d     │    │           │    FE    │     E     │    FE    │     I     │          │           │          │  │
│ │ complaint │  0 │         0 │ 2024-08- │ 2024-08-2 │ 2024-01- │ 2025-03-1 │       19 │         2 │       18 │  │
│ │   _date   │    │           │    28    │     8     │    20    │     8     │          │           │          │  │
│ │           │    │           │ 00:00:00 │ 00:00:00  │ 00:00:00 │ 00:00:00  │          │           │          │  │
│ │ escalatio │  0 │         0 │    N     │     N     │    N     │     Y     │        1 │         1 │        9 │  │
│ │    ns     │    │           │          │           │          │           │          │           │          │  │
│ │  comment  │  5 │ 55.555555 │ service  │ guidance  │ demaned  │  service  │     14.8 │         1 │        9 │  │
│ │           │    │  55555556 │  issue   │ to renew  │ 

In [ ]:
#in customer 
# a. rename name
# b. drop colomns intrest and pincode
# c. change dob datatype
# d. data standardizarion gender
# e. fix missing value in country